# 04_dataset_pipeline.ipynb

## Industrial Dataset Engineering Pipeline

### Objectives

This notebook develops a professional training-ready
dataset pipeline for cardiac ultrasound learning.

Goals:

- train / validation / test dataset engineering
- scalable DataLoader creation
- batching optimization
- worker tuning
- GPU data transfer optimization
- pipeline benchmarking
- stable dataset API validation

In [ ]:
import os
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from torch.utils.data import (
    DataLoader,
    WeightedRandomSampler
)

In [ ]:
%run "./03_preprocessing.ipynb"

In [ ]:
print(EchoDataset)
print(train_transform)
print(FILELIST_PATH)
print(VIDEOS_DIR)

## Configuration

Centralized experiment configuration.

This cell defines:

- batch parameters
- worker settings
- GPU optimization
- reproducibility setup

In [ ]:
CFG = {

    "batch_size":16,
    "num_workers":1,
    "pin_memory":True,
    "persistent_workers":True,
    "shuffle":False,
    "drop_last":False
}

In [ ]:

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(DEVICE)

## Dataset Loading

Load processed metadata generated from
previous preprocessing notebook.

In [ ]:
tracings_df = pd.read_csv(TRACINGS_PATH)
print("Tracings Shape:", tracings_df.shape)
print(tracings_df.head(3))

In [ ]:
df = pd.read_csv(FILELIST_PATH)
print("FileList Shape:", df.shape)
df.head(3)

## Train / Validation / Test Split Construction

Create stable dataset subsets
for downstream training.

In [ ]:
train_df = df[df["Split"]=="TRAIN"].reset_index(drop=True)
val_df = df[df["Split"]=="VAL"].reset_index(drop=True)
test_df = df[df["Split"]=="TEST"].reset_index(drop=True)

print("TRAIN:",len(train_df))
print("VAL:",len(val_df))
print("TEST:",len(test_df))

In [ ]:
fig, ax = plt.subplots( 1, 3,figsize=(16,5))

train_df["EF"].hist(ax=ax[0])
ax[0].set_title("Train EF")
val_df["EF"].hist(ax=ax[1])
ax[1].set_title("Validation EF")
test_df["EF"].hist(ax=ax[2])
ax[2].set_title("Test EF")
plt.show()

## Dataset Construction

Reuse the production-ready
EchoDataset implementation
from 03_preprocessing.

In [ ]:
for split_df in [train_df, val_df, test_df]:
    split_df["Severity"]   = split_df["EF"].apply(ef_to_severity)
    split_df["SeverityID"] = split_df["Severity"].map(SEVERITY_MAP)

print("Train severity distribution:")
print(train_df["Severity"].value_counts())
print("\nVal severity distribution:")
print(val_df["Severity"].value_counts())

In [ ]:
# Distribution check — important before training
print("Train EF stats:")
print(train_df["EF"].describe().round(2))
print("\nVal EF stats:")
print(val_df["EF"].describe().round(2))

# Update CFG with actual train stats — use these for normalization in model
ef_mean = float(train_df["EF"].mean())
ef_std  = float(train_df["EF"].std())
CFG["ef_mean"] = ef_mean
CFG["ef_std"]  = ef_std

print(f"\nEF mean : {ef_mean:.2f}")
print(f"EF std  : {ef_std:.2f}")
print("CFG updated with actual EF stats.")

In [ ]:
train_dataset = EchoDataset(
    filelist=train_df,
    tracings=tracings_df,
    videos_dir=VIDEOS_DIR,
    transform=train_transform
)

val_dataset = EchoDataset(
    filelist=val_df,
    tracings=tracings_df,
    videos_dir=VIDEOS_DIR,
    transform=val_transform
)

test_dataset = EchoDataset(
    filelist=test_df,
    tracings=tracings_df,
    videos_dir=VIDEOS_DIR,
    transform=val_transform
)

In [ ]:
print("Train Dataset:",len(train_dataset))
print("Validation Dataset:",len(val_dataset))
print("Test Dataset:",len(test_dataset))

In [ ]:
print(train_df.columns.tolist())

In [ ]:
train_dataset[0]

In [ ]:
sample = train_dataset[0]

print(type(sample))
print(sample.keys())

## DataLoader Engineering

Create scalable DataLoaders with:

- batching
- worker parallelism
- GPU optimization
- efficient data movement

In [ ]:
def echo_collate_fn(batch):

    return {

        "video": torch.stack(
            [x["video"] for x in batch]
        ),

        "ef": torch.tensor(
            [x["ef"] for x in batch],
            dtype=torch.float32
        ),

        "edv": torch.tensor(
            [x["edv"] for x in batch],
            dtype=torch.float32
        ),

        "esv": torch.tensor(
            [x["esv"] for x in batch],
            dtype=torch.float32
        ),

        "severity": torch.tensor(
            [x["severity"] for x in batch],
            dtype=torch.long
        ),

        "video_name": [
            x["video_name"]
            for x in batch
        ],

        "split": [
            x["split"]
            for x in batch
        ]

    }
    #collate_fn=echo_collate_fn

In [ ]:
from torch.utils.data import RandomSampler

random_sampler = RandomSampler(train_dataset)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [ ]:
print(type(train_loader))
print(len(train_loader))

## Batch Validation

Inspect batch structure,
tensor shapes,
and pipeline compatibility.

In [ ]:
batch = next(
    iter(train_loader)
)

In [ ]:
print(type(batch))

# Inspect Batch Structure

print( "Returned Keys:")
print(batch.keys())
print("Video Batch Shape:",batch["video"].shape)
print("Video Tensor Type:",batch["video"].dtype)

# Label Batch Validation
print("EF Batch:")
print(batch["ef"])
print("Severity Batch:")
print(batch["severity"])

In [ ]:
for key, value in batch.items():
    try:
        print(f"{key:15s} shape: {value.shape} dtype: {value.dtype}")
    except:
        print(f"{key:15s} : {value[:2]}")

In [ ]:
# Extract First Batch Sample

video_batch = batch["video"]
sample_video = video_batch[1]

# Convert Tensors → Images

ed_img = (
    sample_video[0]
    .permute(1,2,0)
    .cpu()
    .numpy()
)

es_img = (
    sample_video[1]
    .permute(1,2,0)
    .cpu()
    .numpy()
)

# Visualize Batch Sample

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow((ed_img - ed_img.min())/(ed_img.max() - ed_img.min()))
plt.title("Batch ED Frame")
plt.axis("off")
plt.subplot(1,2,2)
plt.imshow((es_img - es_img.min())/(es_img.max() - es_img.min()))
plt.title("Batch ES Frame")
plt.axis("off")
plt.tight_layout()
plt.show()

## GPU Pipeline Validation

Validate successful tensor movement
from DataLoader → GPU device.

In [ ]:
for key, value in batch.items():
    if torch.is_tensor(value):
        gpu_tensor = value.to(DEVICE)
        print(f"{key} -> {gpu_tensor.device} | {gpu_tensor.shape}")

## Pipeline Benchmarking

Measure DataLoader throughput
and epoch loading performance.

In [ ]:
loader_iter = iter(train_loader)
start=time.time()
batch = next(loader_iter)
end=time.time()
print(
    "Batch Time:",
    round(end-start,2),
    "sec"
)

In [ ]:
start=time.time()
manual_batch = [
    train_dataset[i]
    for i in range(16)
]
end=time.time()
print(
    "Manual Batch:",
    round(end-start,2),
    "sec"
)

In [ ]:
# Verify EF range in batch is clinically valid
batch = next(iter(train_loader))
ef = batch["ef"]
print(f"Batch EF — min: {ef.min():.1f} max: {ef.max():.1f} mean: {ef.mean():.1f}")
assert ef.min() >= 0 and ef.max() <= 100, "EF out of clinical range"
print("EF range valid.")

## Visualization Validation

Perform final visual sanity checks
on pipeline outputs.

In [ ]:
sample = train_dataset[0]
video = sample["video"]
print(video.shape)

In [ ]:
fig, ax = plt.subplots(
    1,
    2,
    figsize=(10,5)
)

ed = video[0].permute(1,2,0).cpu().numpy()
ed = (ed - ed.min()) / (ed.max() - ed.min())

es = video[1].permute(1,2,0).cpu().numpy()
es = (es - es.min()) / (es.max() - es.min())

ax[0].imshow(ed)
ax[0].set_title("ED Frame")
ax[0].axis("off")

ax[1].imshow(es)
ax[1].set_title("ES Frame")
ax[1].axis("off")

plt.show()

In [ ]:
print("Nonzero Pixels:",(video>0).sum().item())
print("Total Pixels:",video.numel())

print("Min:", video.min().item())
print("Max:", video.max().item())
print("Mean:", video.mean().item())
print("Std:", video.std().item())

In [ ]:
print(batch["video"].shape)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,5))
ed = video[0].permute(1,2,0).cpu().numpy()
ed = (ed - ed.min()) / (ed.max() - ed.min())

es = video[1].permute(1,2,0).cpu().numpy()
es = (es - es.min()) / (es.max() - es.min())

ax[0].imshow(ed)
ax[0].set_title("ED Frame")
ax[0].axis("off")

ax[1].imshow(es)
ax[1].set_title("ES Frame")
ax[1].axis("off")

plt.show()

## Final Pipeline Summary

Completed:

✓ metadata loading

✓ split engineering

✓ Dataset construction

✓ DataLoader engineering

✓ batch validation

✓ GPU transfer validation

✓ pipeline benchmarking

✓ visualization sanity checks

The project now has a stable,
training-ready dataset pipeline.

In [ ]:
print("\n===== 04 PIPELINE SUMMARY =====")
print(f"Train   : {len(train_dataset)} samples | {len(train_loader)} batches")
print(f"Val     : {len(val_dataset)} samples | {len(val_loader)} batches")
print(f"Test    : {len(test_dataset)} samples | {len(test_loader)} batches")
print(f"Batch shape : {batch['video'].shape}")
print(f"Keys        : {list(batch.keys())}")
print("\n04_dataset_pipeline.ipynb executed successfully.")